# Task 06 — Model load

**Wave 1** · target file: `backend/model.py` · prerequisites: none

**🎯 Goal:** `load_models()` / `get_models()` — load the baseline + image artifacts.

Part of the *2026-06-17 fused photo-z backend*. Plan & deps: `../PLAN.md` · conventions: `../README.md`.

In [4]:
# === Setup: run me first. Hops to the repo root so `import backend` and models/ resolve,
# and pulls the model artifact(s) from GCS if missing (needs gcloud auth - see README). ===
import os, sys
p = os.path.abspath('.')
while p != '/' and not (os.path.isdir(p+'/backend') and os.path.isdir(p+'/notebooks')):
    p = os.path.dirname(p)
os.chdir(p); sys.path.insert(0, p)
print('repo root:', p)
os.makedirs('models', exist_ok=True)
if not os.path.exists('models/fake_image_model.pkl'):
    os.system('gcloud storage cp gs://macrocosm-lewagon/models/fake_image_model.pkl models/fake_image_model.pkl')
print('models:', [f for f in os.listdir('models') if f.endswith('.pkl')])

repo root: /Users/mario/code/Macrocosm
models: ['fake_image_model.pkl']


## What to build
In `backend/model.py`:
- **`load_models()`** — `joblib.load(settings.BASELINE_PATH)` (a **dict** `{'model','features',...}` → take
  its `['model']`) and `joblib.load(settings.IMAGE_MODEL_PATH)` (a bare model). Cache both in the
  module globals `_baseline`, `_image`; return `(baseline, image)`.
- **`get_models()`** — return `(baseline, image)`, calling `load_models()` on first use.

> You can dev against a tiny stand-in baseline (below) — no 657MB download needed. In `backend/model.py`
> the functions read `settings.BASELINE_PATH` / `settings.IMAGE_MODEL_PATH` (no arguments).

## 🛠️ Develop & test here first
Fill the `# TODO` so the asserts pass. **Don't** paste the answer — write it.

In [5]:
import joblib, numpy as np, tempfile
from sklearn.linear_model import LinearRegression
from backend.fake_model import RandomRedshiftModel

TMP = tempfile.mkdtemp()
joblib.dump({"model": LinearRegression().fit(np.random.rand(30, 16), np.random.rand(30) * 0.4),
             "features": list(range(16))}, f"{TMP}/baseline.pkl")
joblib.dump(RandomRedshiftModel(0), f"{TMP}/image.pkl")

_baseline = None; _image = None

def load_models(baseline_path=f"{TMP}/baseline.pkl", image_path=f"{TMP}/image.pkl"):
    global _baseline, _image
    _baseline = joblib.load(baseline_path)["model"]
    _image = joblib.load(image_path)
    return _baseline, _image

def get_models():
    global _baseline, _image
    if _baseline is None or _image is None:
        load_models()
    return _baseline, _image

b, i = load_models()
assert hasattr(b, "predict") and hasattr(i, "predict")
assert len(b.predict(np.random.rand(2, 16))) == 2
print("OK", type(b).__name__, "+", type(i).__name__)

OK LinearRegression + RandomRedshiftModel


## ➡️ Move it into `backend/model.py`
Once the cell above passes, put your implementation into **`backend/model.py`** (replace the `# TODO`). Then run the **Check** cell — it imports from `backend/` and verifies the real module.

## ✅ Check (imports from `backend/`)

In [7]:
import os, joblib, numpy as np, tempfile, importlib
from sklearn.linear_model import LinearRegression
from backend.fake_model import RandomRedshiftModel
TMP = tempfile.mkdtemp()
joblib.dump({"model": LinearRegression().fit(np.random.rand(30, 16), np.random.rand(30)), "features": list(range(16))}, f"{TMP}/b.pkl")
joblib.dump(RandomRedshiftModel(0), f"{TMP}/i.pkl")
os.environ["BASELINE_PATH"] = f"{TMP}/b.pkl"; os.environ["IMAGE_MODEL_PATH"] = f"{TMP}/i.pkl"
import backend.config, backend.model
importlib.reload(backend.config); importlib.reload(backend.model)
b, i = backend.model.load_models()
assert hasattr(b, "predict") and hasattr(i, "predict")
print("model-load OK:", type(b).__name__)

model-load OK: LinearRegression


> ⚠️ **06 + 09 both edit `backend/model.py`.** Keep *both* functions when merging — see `MERGE.md`.

## 🚀 Ship it

In [8]:
# === Commit & push on YOUR OWN branch (run last; repo root after Setup) ===
!git checkout -B task/06-model-load
!git add backend/model.py notebooks/tasks-2026-6-17/06-model-load/task.ipynb
!git commit -m "06-model-load: Model load"
!git push -u origin task/06-model-load
# then merge back into 2026.6.17 -> see MERGE.md in this folder

M	backend/model.py
M	notebooks/tasks-2026-6-17/06-model-load/task.ipynb
Switched to a new branch 'task/06-model-load'
[task/06-model-load 12cc5fa] 06-model-load: Model load
 2 files changed, 169 insertions(+), 24 deletions(-)
Enumerating objects: 15, done.
Counting objects: 100% (15/15), done.
Delta compression using up to 10 threads
Compressing objects: 100% (8/8), done.
Writing objects: 100% (8/8), 2.02 KiB | 2.02 MiB/s, done.
Total 8 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
remote: Resolving deltas: 100% (6/6), completed with 6 local objects.
remote: 
remote: Create a pull request for 'task/06-model-load' on GitHub by visiting:
remote:      https://github.com/Le-Wagon-Macrocosm/Macrocosm/pull/new/task/06-model-load
remote: 
To https://github.com/Le-Wagon-Macrocosm/Macrocosm.git
 * [new branch]      task/06-model-load -> task/06-model-load
branch 'task/06-model-load' set up to track 'origin/task/06-model-load'.
